# dbt-for-apache-doris

## 5 个端到端数据工程 Demo

从 Doris Source 开始，依次运行 dbt model、Data Test 和 verifier。每个 Demo 使用独立 Database，可以单独重跑。单元格会展示执行过程、最终状态和结果表；完整原始日志默认折叠，排错时再展开。

| Demo | 主要能力 | 最终对象 |
| --- | --- | --- |
| 每日订单汇总 | Table、Data Test、分区分桶、Async MV | 每日和月度收入 |
| 客户地域分析 | 跨 Database Source、View、`ref()` | 州级客户和收入指标 |
| 广告数据合并 | Seed、`dbt_utils`、`QUALIFY` | 三渠道统一明细 |
| 迟到订单 | Incremental `merge`、Unique Key | 去重后的订单当前版本 |
| 客户 Snapshot | SCD Type 2、Hard Delete | 客户历史和当前维表 |

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
from pathlib import Path
import sys


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "extension/dbt-doris/examples/data-eng-bench-doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 Apache Doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
sys.path.insert(0, str(demo_dir / "scripts"))
from notebook_helpers import DemoRunner

runner = DemoRunner()
runner.show_environment()

## 2. Demo 1：每日订单汇总

**执行链**：`dbt_demo_daily_source.orders` → `daily_order_summary` Table → `monthly_order_summary_mv` Async MV

过滤取消、退货和失败订单，生成每日经营指标，再按月汇总。

In [ ]:
runner.run_demo("每日订单汇总", "data-eng-bench-daily-order-summary")
runner.query("每日订单结果", """
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date
""")
runner.query("月度异步物化视图", """
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv
order by order_month
""")

## 3. Demo 2：客户地域分析

**执行链**：地址 Source + 订单 Source → 2 个 staging View → `fct_state_customers` Table

通过 `source()` 和 `ref()` 读取两个 Doris Database，输出州级客户数、订单数和收入。

In [ ]:
runner.run_demo("客户地域分析", "data-eng-bench-doris-demos/geographic")
runner.query("州级客户与收入", """
select state_province, customer_count, order_count, total_revenue, avg_order_value
from dbt_demo_geographic.fct_state_customers
order by state_province
""")

## 4. Demo 3：广告数据标准化和合并

**执行链**：3 个 CSV → 3 个 Seed Table → 3 个去重 View → `int__ads_unified` Table

统一 Google、Meta 和 TikTok 字段，并用 `dbt_utils` 检查 `source + ad_date` 唯一性。

In [ ]:
runner.run_demo("广告数据标准化和合并", "data-eng-bench-doris-demos/consolidate")
runner.query("统一广告明细", """
select source, ad_date, clicks, impressions, views, conversions
from dbt_demo_consolidate.int__ads_unified
order by source, ad_date
""")

## 5. Demo 4：迟到订单 Incremental

**执行链**：订单事件 → 版本历史 Table → Incremental Unique Key Table → 每日汇总

先全量构建，再写入订单 101 的新版本和新订单 104，执行 `merge`，最后验证无输入变化时结果不漂移。

In [ ]:
runner.run_demo("迟到订单 Incremental", "data-eng-bench-doris-demos/incremental")
runner.query("订单当前版本", """
select order_id, order_date, grand_total, version_num
from dbt_demo_incremental.incremental_daily_sales
order by order_id
""")
runner.query("每日销售汇总", """
select order_date, order_count, total_revenue
from dbt_demo_incremental.daily_sales_summary
order by order_date
""")

## 6. Demo 5：客户 Snapshot

**执行链**：当前客户 Source → staging View → SCD Type 2 Snapshot → 当前客户维表

第二轮修改客户 1 并删除客户 2，验证旧版本关闭、新版本生成和 Hard Delete 失效。

In [ ]:
runner.run_demo("客户 Snapshot", "data-eng-bench-doris-demos/snapshot")
runner.query("客户历史版本", """
select customer_id, email, dbt_valid_from, dbt_valid_to
from dbt_demo_snapshot_history.customer_snapshot
order by customer_id, dbt_valid_from
""")
runner.query("当前客户维表", """
select customer_id, email, total_versions, has_history
from dbt_demo_snapshot.dim_customer_current
order by customer_id
""")

## 完成

5 个运行单元格全部显示绿色“运行通过”，表示对应执行过程、dbt node、Doris 对象和 verifier 均已通过。需要排查编译 SQL 或执行细节时，展开每个结果下方的“查看完整运行日志”。